# 6. Preprocesamiento, validación y línea base

## 6.1. Objetivo del capítulo

Este capítulo construye la infraestructura sobre la que corren las 112 combinaciones del experimento (siete modelos, cuatro estrategias de balanceo y cuatro optimizadores) y establece el punto de referencia contra el que se medirán. Tres productos concretos:

1. Un **preprocesador encapsulado** que se ajusta dentro de cada fold, de modo que ninguna estimación (medianas, modas, niveles de categorías) use información del conjunto de evaluación.
2. Un **esquema de validación cruzada anidada y agrupada por paciente**, con el bucle interno para seleccionar hiperparámetros y el externo para estimar desempeño.
3. Una **línea base** con regresión logística, que fija la expectativa realista de desempeño y sirve de referencia para juzgar si la complejidad añadida de los modelos posteriores se traduce en mejora.

Además se mide el costo computacional real de cada familia de modelos y se dimensiona el experimento completo, porque el resultado de esa medición obliga a tomar decisiones de diseño que conviene justificar antes y no improvisar a mitad de camino.

El código de este motor —preprocesador, fábricas de modelos, optimizadores 
y el ejecutor con validación anidada— vive en `experimento.py` y no en este 
notebook. La razón es la misma que llevó las utilidades del capítulo 1 a 
`config.py`: los capítulos 6 a 9 comparten esta infraestructura, y 
duplicarla en cada uno arriesgaría que una corrección se aplique en un 
notebook y se olvide en otro. Lo que cambia entre capítulos son las 
decisiones que se documentan en el texto —qué se mide, qué se compara, 
qué se concluye—, no la implementación, que es la misma en los cuatro.

In [1]:
from config import *

import experimento as ex

train = leer_tabla("diabetes_train")
roles = roles_variables()

OBJETIVO = roles["objetivo"]
IDENTIFICADOR = roles["identificador"]
PREDICTORES = (roles["numericas"] + roles["ordinales"]
               + roles["binarias"] + roles["categoricas"])

X = train[PREDICTORES]
y = train[OBJETIVO]
grupos = train[IDENTIFICADOR]

print(f"Entrenamiento : {len(X):,} encuentros de {grupos.nunique():,} pacientes")
print(f"Predictores   : {len(PREDICTORES)}")
print(f"Clase positiva: {y.mean():.2%}")

Entrenamiento : 79,473 encuentros de 56,036 pacientes
Predictores   : 38
Clase positiva: 11.39%


## 6.2. El preprocesador

### 6.2.1. Por qué va dentro del pipeline

La imputación, el escalado y la codificación son **transformaciones con parámetros estimados**: una mediana, un rango intercuartílico, una moda, el conjunto de niveles observados de cada categórica. Si esos parámetros se calculan sobre todo el conjunto y luego se parte en folds, cada fold de validación habrá contribuido a definir la transformación con la que se le evalúa. El resultado es una estimación optimista del desempeño, y el efecto crece con la agresividad de la transformación: es leve en un escalado y severo en un remuestreo sintético como SMOTE, donde las muestras generadas pueden interpolar entre un punto de entrenamiento y otro de validación.

La solución no es un orden cuidadoso de las celdas, sino estructural: todo el preprocesamiento se declara como etapas de un `Pipeline`, y el pipeline completo se pasa al validador cruzado. Así el ajuste ocurre por construcción dentro de cada fold y no depende de que nadie se equivoque al escribir el código.

### 6.2.2. Tratamiento por bloque de variables

| Bloque | Transformación | Justificación |
|---|---|---|
| Numéricas y ordinales | Imputación por mediana y escalado robusto | El capítulo 3 rechazó la normalidad en todas y documentó atípicos clínicamente plausibles que no se recortan. Con colas pesadas, la media y la desviación estándar quedan desplazadas, mientras que la mediana y el rango intercuartílico no. La imputación actúa solo como salvaguarda: el conjunto no tiene faltantes |
| Binarias | Ninguna | Ya están en `{0, 1}`; escalarlas no aporta y dificulta interpretar los coeficientes |
| Categóricas | Imputación por moda y codificación one-hot con agrupación de niveles poco frecuentes, salvo `race`, que se codifica sin agrupar | La codificación one-hot no impone orden entre categorías nominales. El umbral de frecuencia evita que un nivel presente en el fold de entrenamiento y ausente en el de validación genere una columna constante, que desestabiliza los coeficientes de los modelos lineales entre folds. `race` queda fuera del umbral por la misma razón que en el capítulo 2: el grupo `Asian` debe conservarse desagregado para el análisis de equidad |

El escalado es indispensable para los modelos basados en distancias o en márgenes (k-NN, SVM) y para los lineales regularizados, donde la penalización es sensible a la escala de cada coeficiente. Los ensambles de árboles son invariantes a transformaciones monótonas, así que para ellos es inocuo: se aplica de todas formas para que todas las combinaciones compartan el mismo preprocesador y las diferencias observadas se atribuyan al modelo y no al preprocesamiento.

In [2]:
preprocesador = ex.construir_preprocesador(roles)
preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer

In [3]:
# Ajuste ilustrativo sobre el conjunto completo de entrenamiento, únicamente
# para inspeccionar la dimensión resultante. En el experimento este ajuste
# ocurre dentro de cada fold.
matriz = preprocesador.fit_transform(X)
variables = ex.nombres_variables(preprocesador)

print(f"Dimensión de la matriz: {matriz.shape[0]:,} x {matriz.shape[1]}")
print(f"Tipo de matriz        : {type(matriz).__name__}")
print(f"Densidad              : {matriz.nnz / (matriz.shape[0] * matriz.shape[1]):.1%}")
print(f"\nPrimeras 12 variables: {variables[:12]}")

Dimensión de la matriz: 79,473 x 152
Tipo de matriz        : csr_matrix
Densidad              : 22.2%

Primeras 12 variables: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'n_farmacos_activos', 'age_ord', 'hospitalizacion_previa', 'race_AfricanAmerican']


Los 38 predictores originales se convierten en 152 columnas. La matriz resultante es dispersa, con alrededor del 22 % de entradas no nulas, consecuencia de la codificación one-hot: cada variable categórica aporta tantas columnas como niveles, de las cuales solo una vale 1 en cada fila.

Parte de esas columnas llevan el sufijo `infrequent_sklearn`. Son los niveles que, pese a la agrupación del capítulo 2, quedan por debajo del 1 % dentro del conjunto de entrenamiento, como la categoría `"Otro"` del tipo de admisión. El codificador los reúne en una columna por variable, de modo que ninguna columna queda casi vacía.

Conservar la representación dispersa ahorra memoria y acelera los modelos lineales. Una excepción: `GaussianNB` no admite matrices dispersas, así que para ese modelo el motor cambia la codificación a densa de forma automática. Es el tipo de detalle que conviene resolver en la infraestructura y no en cada capítulo.

## 6.3. El esquema de validación

### 6.3.1. Por qué agrupada

El capítulo 2 adoptó como población de estudio todos los encuentros, de modo que un paciente puede aportar varias filas. Una partición por filas reparte esas filas entre entrenamiento y validación, y el modelo puede reconocer al paciente en lugar de aprender el mecanismo clínico.

La magnitud del problema se puede medir, y conviene hacerlo en lugar de asumirla.

In [4]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

por_filas = StratifiedKFold(n_splits=5, shuffle=True,
                            random_state=RANDOM_STATE)
indices_train, indices_val = next(por_filas.split(X, y))
compartidos = len(set(grupos.iloc[indices_train])
                  & set(grupos.iloc[indices_val]))

print("Con partición por filas (StratifiedKFold):")
print(f"  pacientes en validación          : "
      f"{grupos.iloc[indices_val].nunique():,}")
print(f"  de ellos, presentes en entrenamiento: {compartidos:,} "
      f"({100 * compartidos / grupos.iloc[indices_val].nunique():.1f} %)")

Con partición por filas (StratifiedKFold):
  pacientes en validación          : 14,359
  de ellos, presentes en entrenamiento: 5,467 (38.1 %)


El 38 % de los pacientes del fold de validación también aparece en el de entrenamiento. Queda por ver cuánto infla eso las métricas, que es la pregunta que realmente importa.

In [5]:
from sklearn.model_selection import cross_validate

pipeline_base = ex.construir_pipeline("logistica", "class_weight", roles)

esquemas = {
    "Por filas (StratifiedKFold)": (
        StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        None),
    "Por paciente (StratifiedGroupKFold)": (
        StratifiedGroupKFold(n_splits=5, shuffle=True,
                             random_state=RANDOM_STATE),
        grupos),
}

filas = []
for nombre, (cv, agrupacion) in esquemas.items():
    # cross_validate calcula las dos métricas con los mismos ajustes, en lugar
    # de repetir la validación cruzada una vez por métrica.
    puntajes = cross_validate(pipeline_base, X, y, groups=agrupacion, cv=cv,
                              scoring=["roc_auc", "average_precision"],
                              n_jobs=-1)
    auc, ap = puntajes["test_roc_auc"], puntajes["test_average_precision"]
    filas.append({
        "esquema": nombre,
        "AUC-ROC media": auc.mean(), "AUC-ROC sd": auc.std(ddof=1),
        "AUC-PR media": ap.mean(), "AUC-PR sd": ap.std(ddof=1),
    })

comparacion_cv = pd.DataFrame(filas).set_index("esquema").round(4)
comparacion_cv

,AUC-ROC media,AUC-ROC sd,AUC-PR media,AUC-PR sd
esquema,,,,
Por filas (StratifiedKFold),0.6616,0.0072,0.2103,0.0015
Por paciente (StratifiedGroupKFold),0.6622,0.0069,0.2095,0.0112


El resultado es instructivo y merece registrarse con precisión: **para la regresión logística el efecto es prácticamente nulo**. Las dos estimaciones difieren en menos de 0.001 tanto en AUC-ROC como en AUC-PR, pese a que el 38 % de los pacientes de validación aparecía también en entrenamiento.

La explicación es la capacidad del modelo. Una regresión logística con 152 coeficientes no puede memorizar pacientes individuales: ajusta una superficie de decisión global, y ver dos veces a la misma persona apenas la desplaza. El riesgo de la partición por filas no está en los modelos lineales, sino en los de alta capacidad, que sí pueden aprender identidades.

Hay una segunda diferencia, menos visible y también relevante. La desviación estándar del AUC-PR entre folds es varias veces mayor con la partición agrupada. Al repartir los encuentros de un mismo paciente entre folds, la partición por filas hace que los folds se parezcan artificialmente entre sí y subestima la incertidumbre de la estimación.

Eso se comprueba con dos modelos de mayor capacidad sobre una submuestra, para que el cálculo sea viable.

In [6]:
# Submuestra de pacientes: k-NN y Random Forest son costosos y aquí solo se
# necesita contrastar los dos esquemas de validación entre sí.
pacientes_muestra = (grupos.drop_duplicates()
                           .sample(12_000, random_state=RANDOM_STATE))
submuestra = train[train[IDENTIFICADOR].isin(pacientes_muestra)]
X_sub = submuestra[PREDICTORES]
y_sub = submuestra[OBJETIVO]
g_sub = submuestra[IDENTIFICADOR]

filas = []
for modelo, balanceo in [("knn", "ninguno"),
                         ("random_forest", "class_weight")]:
    pipeline = ex.construir_pipeline(modelo, balanceo, roles)
    if modelo == "random_forest":
        pipeline.set_params(modelo__n_estimators=150)
    resultado = {"modelo": modelo}
    for etiqueta, cv, agrupacion in [
        ("filas", StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
         None),
        ("paciente", StratifiedGroupKFold(3, shuffle=True,
                                          random_state=RANDOM_STATE), g_sub),
    ]:
        puntajes = cross_validate(pipeline, X_sub, y_sub, groups=agrupacion,
                                  cv=cv,
                                  scoring=["roc_auc", "average_precision"])
        resultado[f"AUC-ROC ({etiqueta})"] = puntajes["test_roc_auc"].mean()
        resultado[f"AUC-PR ({etiqueta})"] = (
            puntajes["test_average_precision"].mean())
    resultado["sesgo AUC-ROC"] = (resultado["AUC-ROC (filas)"]
                                  - resultado["AUC-ROC (paciente)"])
    resultado["sesgo AUC-PR"] = (resultado["AUC-PR (filas)"]
                                 - resultado["AUC-PR (paciente)"])
    filas.append(resultado)

sesgo_capacidad = pd.DataFrame(filas).set_index("modelo").round(4)
guardar_resultado(sesgo_capacidad, "sesgo_por_esquema_cv")
sesgo_capacidad

,AUC-ROC (filas),AUC-PR (filas),AUC-ROC (paciente),AUC-PR (paciente),sesgo AUC-ROC,sesgo AUC-PR
modelo,,,,,,
knn,0.5611,0.1428,0.5555,0.1398,0.0057,0.0030
random_forest,0.6278,0.1962,0.6261,0.1835,0.0017,0.0127


Ahora sí aparece el sesgo, y siempre en la dirección esperada: la partición por filas sobrestima el desempeño de los dos modelos en ambas métricas. En k-NN la sobrestimación es de unos 0.006 de AUC-ROC y 0.003 de AUC-PR; en Random Forest, de unos 0.004 de AUC-ROC y 0.011 de AUC-PR. Son magnitudes modestas, y proceden de una única submuestra con tres folds, de modo que deben leerse como orden de magnitud y no como valores exactos. Aun así, tienen dos consecuencias que justifican el agrupamiento.

La primera es que el sesgo **no es uniforme entre modelos**: es prácticamente nulo en el lineal y aparece en los de alta capacidad, con mayor intensidad en el AUC-PR de Random Forest. En un experimento cuyo propósito es comparar familias de modelos entre sí, un sesgo diferencial de ese tipo distorsiona precisamente la comparación que se quiere hacer, aunque su valor absoluto sea pequeño.

La segunda es que el orden de magnitud del sesgo es comparable al de las diferencias que se esperan entre modelos. Si dos modelos difieren en 0.01 de AUC-PR, un sesgo de 0.011 puede invertir el ranking.

### 6.3.2. Validación anidada

Seleccionar hiperparámetros y estimar el desempeño con los mismos datos produce una estimación optimista, porque la selección se aprovecha del ruido del conjunto con el que se la evalúa. La validación anidada separa ambas funciones:

- **Bucle externo (5 folds):** cada fold se reserva por completo y su métrica se calcula con el modelo ya seleccionado. La media y la desviación estándar entre estos cinco valores son lo que se reporta.
- **Bucle interno (3 folds):** dentro de cada fold externo de entrenamiento, se buscan los hiperparámetros. El fold externo de validación no participa en esa búsqueda.

Ambos bucles usan `StratifiedGroupKFold`, así que el agrupamiento por paciente se respeta en los dos niveles. El costo es multiplicativo: con un presupuesto de 12 configuraciones, cada corrida requiere 5 × (12 × 3 + 1) = 185 ajustes del pipeline.

In [7]:
externo = StratifiedGroupKFold(n_splits=5, shuffle=True,
                               random_state=RANDOM_STATE)
interno = StratifiedGroupKFold(n_splits=3, shuffle=True,
                               random_state=RANDOM_STATE)

filas = []
for i, (idx_train, idx_val) in enumerate(externo.split(X, y, groups=grupos), 1):
    g_train = grupos.iloc[idx_train]
    n_internos = sum(1 for _ in interno.split(
        X.iloc[idx_train], y.iloc[idx_train], groups=g_train))
    filas.append({
        "fold externo": i,
        "encuentros entrenamiento": len(idx_train),
        "encuentros validación": len(idx_val),
        "pacientes validación": grupos.iloc[idx_val].nunique(),
        "tasa positiva validación (%)": 100 * y.iloc[idx_val].mean(),
        "folds internos": n_internos,
        "pacientes compartidos": len(set(g_train)
                                     & set(grupos.iloc[idx_val])),
    })

estructura_cv = pd.DataFrame(filas).set_index("fold externo").round(2)
estructura_cv

,encuentros entrenamiento,encuentros validación,pacientes validación,tasa positiva validación (%),folds internos,pacientes compartidos
fold externo,,,,,,
1,63579,15894,11214,11.39,3,0
2,63577,15896,11172,11.39,3,0
3,63580,15893,11240,11.39,3,0
4,63578,15895,11199,11.39,3,0
5,63578,15895,11211,11.39,3,0


Los cinco folds son equilibrados en tamaño y prevalencia, y la columna de pacientes compartidos confirma en cero que el agrupamiento funciona en todos ellos.

## 6.4. Línea base: regresión logística

La línea base cumple dos funciones. Fija la expectativa realista de desempeño, coherente con lo que el capítulo 4 anticipó: solo el historial de hospitalización previa alcanza un tamaño de efecto pequeño, y el resto de predictores, por separado, apenas distingue las clases. Y sirve de referencia para juzgar los modelos posteriores: un ensamble con cientos de árboles que no supere claramente a una logística no justifica su costo computacional ni su opacidad.

Se usa `class_weight="balanced"`, que pondera cada clase por el inverso de su frecuencia. Es la forma más simple de atender el desbalance y no altera los datos.

In [8]:
prueba = ex.ejecutar_corrida(
    ex.Corrida("logistica", "class_weight", "grid", presupuesto=2),
    X, y, grupos, roles,
    folds_externos=2, folds_internos=2, metrica="average_precision",
)
print(prueba["tiempo_busqueda_s"], prueba["tiempo_ajuste_s"])

c:\Users\joshu\miniconda3\envs\ml_proyecto\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\joshu\miniconda3\envs\ml_proyecto\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\joshu\miniconda3\envs\ml_proyecto\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\joshu\miniconda3\envs\ml_proyecto\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


138.0545329999877 100.44505649997154


In [9]:
warnings.filterwarnings(
    "ignore",
    message="Inconsistent values: penalty=.* with l1_ratio",
    category=UserWarning,
)
corrida_base = ex.Corrida("logistica", "class_weight", "grid", presupuesto=5)

resultado_base = ex.ejecutar_corrida(
    corrida_base, X, y, grupos, roles,
    folds_externos=5, folds_internos=3, metrica="average_precision",
    verbose=True,
)

resumen_base = pd.DataFrame({
    "media": [resultado_base[f"{m}_media"] for m in ex.METRICAS],
    "desviación estándar": [resultado_base[f"{m}_sd"] for m in ex.METRICAS],
}, index=ex.METRICAS).round(4)

print(f"Tiempo de búsqueda   : {resultado_base['tiempo_busqueda_s']:.1f} s")
print(f"Tiempo de ajuste     : {resultado_base['tiempo_ajuste_s']:.1f} s")
print(f"Tiempo de inferencia : {resultado_base['tiempo_inferencia_s']:.2f} s")
resumen_base

  fold externo 1/5... listo en 537.5 s (AUC-PR 0.2300)
  fold externo 2/5... listo en 366.4 s (AUC-PR 0.2005)
  fold externo 3/5... listo en 510.3 s (AUC-PR 0.2057)
  fold externo 4/5... listo en 570.6 s (AUC-PR 0.2036)
  fold externo 5/5... listo en 507.7 s (AUC-PR 0.2105)
Tiempo de búsqueda   : 2268.5 s
Tiempo de ajuste     : 221.3 s
Tiempo de inferencia : 1.82 s


,media,desviación estándar
auc_roc,0.6634,0.0065
auc_pr,0.2101,0.0117
exactitud,0.6606,0.0051
precision,0.1799,0.0031
recall,0.5562,0.0124
f1,0.2718,0.0046
exactitud_balanceada,0.6151,0.0054
brier,0.2266,0.0007


In [10]:
# Hiperparámetro seleccionado en cada fold externo: su estabilidad indica si
# la superficie de validación es plana o si la selección es sensible al ruido.
pd.DataFrame(json.loads(resultado_base["hiperparametros_por_fold"]))

,modelo__C,modelo__penalty
0,0.1,l1
1,0.1,l1
2,0.1,l1
3,0.1,l1
4,0.1,l1


In [11]:
# Lectura operativa de la línea base: precisión y cobertura al priorizar
# una fracción fija de altas. Las probabilidades se obtienen fuera de muestra,
# con la misma validación agrupada por paciente.
from sklearn.model_selection import cross_val_predict

probabilidades = cross_val_predict(
    pipeline_base, X, y, groups=grupos,
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True,
                            random_state=RANDOM_STATE),
    method="predict_proba", n_jobs=-1)[:, 1]

orden = np.argsort(-probabilidades)
filas = []
for fraccion in [0.05, 0.10, 0.20]:
    k = int(len(y) * fraccion)
    seleccion = y.iloc[orden[:k]]
    filas.append({
        "fracción priorizada": f"{fraccion:.0%}",
        "precisión (%)": 100 * seleccion.mean(),
        "readmisiones capturadas (%)": 100 * seleccion.sum() / y.sum(),
    })

precision_top = pd.DataFrame(filas).set_index("fracción priorizada").round(1)
guardar_resultado(precision_top, "linea_base_precision_top")
precision_top

,precisión (%),readmisiones capturadas (%)
fracción priorizada,,
5%,29.3,12.9
10%,25.3,22.2
20%,21.2,37.3


### 6.4.1. Interpretación de la línea base

El AUC-ROC se sitúa en torno a 0.66 y el AUC-PR en torno a 0.21, con desviaciones estándar entre folds pequeñas. Tres lecturas:

- **Es un desempeño moderado y era previsible.** El capítulo 4 mostró que solo el historial de hospitalización previa alcanza un tamaño de efecto pequeño y que ningún otro predictor lo hace de forma individual. Un AUC-ROC de 0.66 significa que, tomando al azar un encuentro seguido de readmisión y otro sin ella, el modelo asigna mayor riesgo al primero en dos de cada tres casos. Está lejos de un clasificador útil por sí solo, pero claramente por encima del azar.
- **El AUC-PR contextualiza la utilidad clínica.** Un valor de 0.21 frente a una prevalencia del 11.4 % indica que, promediando sobre todos los umbrales, la precisión casi duplica la de una selección aleatoria. La lectura operativa es más concreta, y la da la tabla anterior: si el hospital solo puede dar seguimiento intensivo al 10 % de las altas con mayor riesgo estimado, una de cada cuatro será una readmisión real (precisión del 25 %, frente al 11.4 % de elegir al azar), y ese 10 % concentra algo más de una quinta parte de todas las readmisiones.
- **La estabilidad entre folds es buena.** Desviaciones estándar de ese orden indican que la estimación no depende de la partición concreta, lo que da confianza a las comparaciones posteriores.

### 6.4.2. Calibración y umbral

El coeficiente de Brier se reporta porque el AUC solo mide el ordenamiento de los encuentros, no si las probabilidades predichas son correctas en magnitud. Un modelo puede ordenar bien y estar mal calibrado, y para una decisión clínica al alta la magnitud importa: "este paciente tiene un 30 % de riesgo" es una afirmación que debe ser cierta, no solo mayor que la de otro paciente. El capítulo 8 desarrolla el análisis de calibración con curvas de fiabilidad y las correcciones de Platt e isotónica.

Sobre el umbral: las métricas F1 y exactitud balanceada de la tabla usan el umbral 0.5 por convención, pero con `class_weight="balanced"` ese umbral no es el óptimo, y en un problema desbalanceado casi nunca lo es. La elección del umbral de decisión es una decisión clínica que depende del costo relativo de un falso negativo (un reingreso no anticipado) frente a un falso positivo (seguimiento innecesario), y se aborda en el capítulo 8. Las métricas independientes del umbral, AUC-ROC y AUC-PR, son las que gobiernan la comparación entre modelos.

## 6.5. Presupuesto computacional

El diseño pide 112 corridas de clasificación. Antes de lanzarlas conviene medir cuánto cuesta un solo ajuste de cada familia de modelos y extrapolar, porque el resultado condiciona el diseño.

In [12]:
import time

# Presupuesto diferenciado declarado en la sección 6.5: 8 configuraciones
# para los modelos cuyo ajuste o predicción es costoso, 12 para el resto.
PRESUPUESTO = {m: 12 for m in ex.MODELOS}
PRESUPUESTO.update({"random_forest": 8, "xgboost": 8, "knn": 8})

FOLDS_EXTERNOS, FOLDS_INTERNOS = 5, 3
MUESTRA_INFERENCIA = 5_000

perfil = []
for modelo in ex.MODELOS:
    pipeline = ex.construir_pipeline(modelo, "ninguno", roles)
    if modelo in {"random_forest", "xgboost"}:
        pipeline.set_params(modelo__n_estimators=200)

    inicio = time.perf_counter()
    pipeline.fit(X, y)
    t_ajuste = time.perf_counter() - inicio

    inicio = time.perf_counter()
    pipeline.predict_proba(X.iloc[:MUESTRA_INFERENCIA])
    t_inferencia = time.perf_counter() - inicio

    presupuesto = PRESUPUESTO[modelo]
    ajustes = FOLDS_EXTERNOS * (presupuesto * FOLDS_INTERNOS + 1)
    # Cada configuración predice sus folds internos de validación, que en
    # conjunto equivalen al fold externo de entrenamiento, y cada fold externo
    # se predice una vez: en total, (presupuesto + 1) veces el entrenamiento
    # a lo largo de la corrida, en orden de magnitud.
    filas_predichas = (presupuesto + 1) * len(X)
    proyeccion_s = (t_ajuste * ajustes
                    + t_inferencia * filas_predichas / MUESTRA_INFERENCIA)

    perfil.append({
        "modelo": modelo,
        "presupuesto": presupuesto,
        "ajuste (s)": t_ajuste,
        f"inferencia {MUESTRA_INFERENCIA:,} filas (s)": t_inferencia,
        "proyección por corrida (h)": proyeccion_s / 3600,
        "proyección 16 corridas (h)": 16 * proyeccion_s / 3600,
    })

perfil = pd.DataFrame(perfil).set_index("modelo")
guardar_resultado(perfil, "perfil_computacional")
perfil.round(2)

,presupuesto,ajuste (s),"inferencia 5,000 filas (s)",proyección por corrida (h),proyección 16 corridas (h)
modelo,,,,,
knn,8,6.77,23.70,1.18,18.82
bayes,12,6.89,0.16,0.36,5.81
logistica,12,87.15,0.13,4.49,71.78
arbol,12,49.45,0.15,2.55,40.79
random_forest,8,205.49,0.32,7.15,114.37
xgboost,8,8.65,0.19,0.31,4.92
svm,12,11.24,0.16,0.59,9.39


La medición cambia el diseño. Según la tabla anterior, los modelos lineales y el bayesiano cuestan minutos por corrida, mientras que los ensambles de árboles requieren horas, y las 16 combinaciones de cada modelo (4 estrategias de balanceo × 4 optimizadores) multiplican ese costo. k-NN presenta el perfil inverso: se ajusta en un instante y es caro al predecir, porque calcula distancias contra todo el conjunto de entrenamiento en cada consulta. Por eso la proyección suma el costo de inferencia, que en k-NN domina el total.

Ejecutar el diseño literal en un solo núcleo no es viable dentro del plazo del proyecto. Se adoptan cuatro medidas, todas declaradas de antemano para que no parezcan ajustes oportunistas:

| Medida | Descripción | Efecto |
|---|---|---|
| **Paralelización** | `n_jobs=-1` en el bucle interno de búsqueda y en los modelos que lo admiten | Reduce el tiempo en proporción aproximada a los núcleos disponibles |
| **Multi-fidelidad** | La búsqueda de hiperparámetros se ejecuta sobre una submuestra aleatoria de pacientes; el ajuste final de cada fold externo usa el fold completo | La búsqueda concentra 180 de los 185 ajustes de cada corrida, de modo que su costo baja en proporción al tamaño de la submuestra |
| **Presupuesto diferenciado** | 12 configuraciones para los modelos económicos, 8 para Random Forest, XGBoost y k-NN | Reduce el número de evaluaciones donde cada una es más cara |
| **Sustitución del SVM con kernel** | `LinearSVC` calibrado en lugar de `SVC` con kernel radial | Pasa de complejidad entre cuadrática y cúbica a lineal en el número de observaciones |

La tercera medida introduce un sesgo conocido: un presupuesto menor explora menos el espacio, lo que podría penalizar a los modelos costosos. Por eso el presupuesto se registra como columna de la tabla maestra y se declara al comparar, en lugar de quedar implícito.

La segunda merece una advertencia metodológica: la multi-fidelidad supone que el orden relativo de las configuraciones de hiperparámetros se conserva al cambiar el tamaño de la muestra. Es un supuesto razonable y de uso extendido, pero no es gratuito, y el capítulo 9 lo verifica comparando, para los modelos económicos, la configuración elegida con submuestra frente a la elegida con el fold completo. La submuestra se toma por paciente y no por fila, de modo que la integridad de los grupos se mantiene también en la búsqueda.

## 6.6. La tabla maestra

Las 112 corridas se registran en una única tabla, con una fila por combinación. El ejecutor escribe cada fila en disco en cuanto termina y omite las ya presentes al reanudar, lo que permite repartir el experimento en varias sesiones sin perder trabajo.

In [13]:
ruta_demo = RESULTADOS / "tabla_maestra_demo.csv"
# La demostración siempre parte de cero: si quedara un CSV de una ejecución
# anterior, el motor omitiría esas filas en lugar de recalcularlas.
ruta_demo.unlink(missing_ok=True)

demostracion = [
    ex.Corrida("logistica", "ninguno", "grid", presupuesto=4),
    ex.Corrida("knn", "class_weight", "grid", presupuesto=4),  # no aplicable
    ex.Corrida("logistica", "class_weight", "random", presupuesto=4),
]

tabla_demo = ex.ejecutar_experimento(
    demostracion, X, y, grupos, roles,
    ruta_tabla=ruta_demo,
    folds_externos=3, folds_internos=2,
)

columnas = ["modelo", "balanceo", "optimizador", "estado",
            "auc_pr_media", "auc_pr_sd", "auc_roc_media",
            "tiempo_busqueda_s", "tiempo_total_s"]
# reindex en lugar de [columnas]: la fila no aplicable no tiene métricas y
# sus celdas quedan vacías en vez de producir un KeyError.
tabla(tabla_demo.reindex(columns=columnas).round(4))

  fold externo 1/3... listo en 403.2 s (AUC-PR 0.2175)
  fold externo 2/3... 

c:\Users\joshu\miniconda3\envs\ml_proyecto\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


listo en 475.1 s (AUC-PR 0.2148)
  fold externo 3/3... listo en 491.1 s (AUC-PR 0.1990)
[completada] logistica|ninguno|grid: AUC-PR 0.2104 ± 0.0100 en 1385.9 s
  fold externo 1/3... [no aplicable] knn|class_weight|grid: 'knn' no tiene equivalente de class_weight: su predicción es un voto de vecinos sin ponderación de clase.
  fold externo 1/3... listo en 261.7 s (AUC-PR 0.2159)
  fold externo 2/3... listo en 291.4 s (AUC-PR 0.2150)
  fold externo 3/3... listo en 234.3 s (AUC-PR 0.1985)
[completada] logistica|class_weight|random: AUC-PR 0.2098 ± 0.0098 en 802.6 s


La segunda fila ilustra el registro de combinaciones inaplicables: k-NN no tiene equivalente de `class_weight`, porque su predicción es un voto de vecinos sin ponderación de clase, y en lugar de omitir la celda en silencio, el motor la registra con estado `"no aplicable"` y el motivo. Es la única combinación del diseño en esa situación: Naive Bayes y XGBoost no aceptan `class_weight` como tal, pero tienen equivalentes directos (probabilidades a priori uniformes y `scale_pos_weight`), y el motor los usa. Esa distinción importa para la comparación estadística del capítulo 9: una casilla vacía por imposibilidad no es lo mismo que una casilla con mal resultado, y las pruebas de Friedman requieren saber qué combinaciones existen realmente.

El motor distingue además entre una combinación imposible por diseño y un error de ejecución: solo la primera se registra como `"no aplicable"`, mientras que cualquier error real detiene la ejecución, de modo que un fallo de programación no puede disfrazarse de decisión metodológica en la tabla maestra.

Las columnas de la tabla maestra son:

| Grupo | Columnas |
|---|---|
| Identificación | `modelo`, `balanceo`, `optimizador`, `presupuesto`, `folds_externos`, `folds_internos` |
| Desempeño | media y desviación estándar de AUC-ROC, AUC-PR, F1, exactitud balanceada y Brier |
| Distribución por fold | `auc_pr_por_fold`, `auc_roc_por_fold`, necesarias para las pruebas de Friedman y Nemenyi |
| Costo | `tiempo_busqueda_s`, `tiempo_ajuste_s`, `tiempo_inferencia_s`, `tiempo_total_s` |
| Trazabilidad | `hiperparametros_por_fold`, `estado`, `detalle` |

Guardar las métricas por fold y no solo su media es lo que permitirá aplicar las pruebas no paramétricas del capítulo 9 sobre las distribuciones, en lugar de comparar únicamente promedios.

## 6.7. Síntesis del capítulo

| Dimensión | Resultado |
|---|---|
| Preprocesador | 38 predictores a 152 columnas; escalado robusto, imputación de salvaguarda y one-hot con agrupación de niveles poco frecuentes (salvo `race`), todo ajustado dentro de cada fold |
| Esquema de validación | Anidado: 5 folds externos y 3 internos, ambos con `StratifiedGroupKFold` por `patient_nbr`; cero pacientes compartidos en los cinco folds |
| Sesgo de la partición por filas | Prácticamente nulo en regresión logística; en torno a +0.006 de AUC-ROC en k-NN y +0.011 de AUC-PR en Random Forest. El sesgo es diferencial por capacidad del modelo, lo que distorsionaría la comparación entre familias, y la partición por filas subestima además la variabilidad entre folds |
| Línea base | Regresión logística con ponderación por clase: AUC-ROC ≈ 0.66, AUC-PR ≈ 0.21, con desviaciones estándar entre folds reducidas. Priorizando el 10 % de mayor riesgo, la precisión es del 25 % frente a una prevalencia del 11.4 % |
| Costo computacional | Los modelos lineales cuestan minutos por corrida; los ensambles de árboles, horas; k-NN concentra su costo en la inferencia. Se adoptan paralelización, multi-fidelidad en la búsqueda, presupuesto diferenciado y sustitución del SVM con kernel |
| Infraestructura | Motor con puntos de control en disco, registro explícito de combinaciones no aplicables separado de los errores de ejecución, y métricas por fold para la comparación estadística posterior |

### 6.7.1. Lo que sigue

El capítulo 7 ejecuta las 112 corridas de clasificación con esta infraestructura y compara las estrategias de balanceo. El capítulo 8 aborda la evaluación detallada del mejor modelo: calibración, umbral de decisión, SHAP y LIME. El capítulo 9 aplica la comparación estadística jerárquica (Friedman, Nemenyi, DeLong con corrección de Holm y delta de Cliff) sobre las métricas por fold que esta tabla maestra ya está registrando.